# Baseline zero-shot sobre los 150 casos del D3 · Phi-3.5-mini

La tabla del documento tenía el baseline zero-shot solo en los 180 sellados. Los 60 originales
ya lo tienen desde el D1 (`resultados/Phi-3.5-mini-instruct__zero_shot__prosa.raw.jsonl`); este
notebook corre los 60 nuevos y las 30 redactadas con el mismo `runner.py` y la misma condición
que la réplica. Corrido el 31 de octubre, después de las demás mediciones. Es el baseline, no el
sistema: no toca ningún protocolo.

Requiere GPU T4. Corre las celdas en orden, de arriba abajo (unos 20 a 30 minutos).

In [ ]:
# 1 · repositorio, dependencias y GPU
COMMIT = '56ba839'

%cd /content
!nvidia-smi -L
!pip -q install transformers==5.17.0 accelerate==1.14.0 bitsandbytes==0.50.2
!rm -rf /content/proyecto && git clone -q https://github.com/Luquitas02/genai-inscripcion-ramos /content/proyecto
import subprocess
subprocess.run(['git', '-C', '/content/proyecto', 'checkout', '-q', COMMIT], check=True)
!git -C /content/proyecto log -1 --format='commit %h  %ad  %s'
import torch; print('cuda:', torch.cuda.is_available())
from importlib.metadata import version
for paquete in ('torch', 'transformers', 'accelerate', 'bitsandbytes'):
    print(f'{paquete}=={version(paquete)}')
from huggingface_hub import model_info
REVISION = model_info('microsoft/Phi-3.5-mini-instruct').sha
print('revisión del modelo en Hugging Face:', REVISION)
assert REVISION.startswith('2fe1924'), '*** otra revisión del modelo: NO SIGAS, avisa ***'

In [ ]:
# 2 · autopruebas sin GPU
import sys, os
os.chdir('/content/proyecto/scripts')
for args in (['verificador.py'], ['herramientas.py']):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    print(f'[{"OK  " if r.returncode == 0 else "FALLA"}] {args[0]:18} {ultima[:70]}')
    assert r.returncode == 0, '*** NO SIGAS: falló ' + args[0] + ' ***'
print('TODO EN ORDEN')

In [ ]:
# 3 · zero-shot sobre los 60 nuevos y las 30 redactadas (cada uno carga el modelo)
from pathlib import Path
for conjunto in ('casos_nuevos.jsonl', 'casos_redactados.jsonl'):
    r = subprocess.run([sys.executable, 'runner.py', '--modelo', 'microsoft/Phi-3.5-mini-instruct',
                        '--condicion', 'zero_shot', '--casos', f'../datos/{conjunto}'],
                       capture_output=True, text=True)
    print(f'===== {conjunto}')
    print('
'.join(r.stdout.strip().splitlines()[-12:]))
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit('*** falló la corrida de ' + conjunto + ' ***')
print('zero-shot completo')

In [ ]:
# 4 · bajar los dos archivos de resultados
from google.colab import files
for conjunto in ('casos_nuevos', 'casos_redactados'):
    files.download(f'/content/proyecto/resultados/Phi-3.5-mini-instruct__zero_shot__prosa__{conjunto}.raw.jsonl')